##### ***主流的微调手段 SFT(supervised Fine-tuning)***
###### 如果说LoRA是决定微调时参数如何变化，那么SFT则决定微调的具体手段已经要达到什么效果
###### 举个具体例子：一个只经过预训练的模型，例如GPT-2，已经具备了较强的语言建模能力。它可以根据前文流畅地续写文本，但这种能力并不等同于"听指令"。当用户给它一段提问时，base model并不会天然地按照提问去组织回答，而更可能只是顺着输入继续自由发挥。
###### 如果直接让一个base model生成，它也许会在看到"User: 什么是注意力机制？"之后，继续编造出一段无关的对话，而不是给出一个明确的定义。原因在于，预训练阶段模型见过的绝大多数文本都是自然语料，而不是"提问-回答"这样结构化的对话。因此，模型虽然"会说话"，却没有学会"该怎么应对别人的问题"。
###### SFT要解决的就是这个问题。通过提供大量"指令-回答"配对的数据，并让模型在监督信号下模仿这些回答，模型会逐渐调整自己的行为，学会在看到提问时按照期望的格式与内容进行回答。经过这个过程，一个base model便慢慢转变为一个instruct model，也就是一个更接近助手的模型。
###### 所以简而言之，SFT本质是利用较少的，人类构造的一类特定数据集，来使模型的行为对齐到某种预期回答上。从逻辑上来看，其实SFT与正常数据的训练并无差别，区别仅仅是数据集，对SFT来说，数据的形式往往是问答的形式，而为了学习输出的形式，在具体计算时，只需要计算预期回答的部分与实际回答部分的交叉熵即可，因为我们的目的是让模型学会输出而不是学会提问。所以将prompt token从loss中mask掉，只计算response token计算语言建模损失。
###### 举个SFT数据的例子：<br>User: 什么是注意力机制？<br>Assistant: 注意力机制是一种让模型在处理序列时关注输入中相关部分的方法。

##### ***代码实现***
###### 为方便实现，这里使用github的开源工具库TRL进行实现，官网文档在：https://github.com/huggingface/trl
###### TRL中的实现SFT的关键类是SFTTrainer，相比于手写编写完整的训练循环，SFTTrainer对SFT过程中常见的数据预处理、批处理、前向传播、损失计算、反向传播、参数更新、日志记录以及模型保存等流程进行了封装，因此在实际使用中只需要提供模型、训练数据以及训练参数即可完成基本的SFT训练。
###### 在正式构建之前，首先需要准备要进行微调的Base Model以及对应的Tokenizer。通常可以直接通过Hugging Face Transformers中的AutoModelForCausalLM和AutoTokenizer进行加载：

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, PeftModel
import torch

# 加载基础模型
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B",
    torch_dtype = "auto",
    device_map = "auto"
)

# 在SFT中Tokenizer应该与模型相互匹配，因为不同的模型通常具有不同的词表、特殊token以及Chat Template。所以Tokenizer与模型不匹配，可能导致输入格式错误甚至模型无法正常训练
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

# dataset加载(ChatML格式)
dataset = load_dataset("trl-lib/Capybara", split="train")

"""
ChatML格式如下：
{
    "messages": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."}
    ]
}

"""
# LoRA配置
lora_config = LoraConfig(
    r = 64,
    lora_alpha=128,
    # target_modules表示在哪些层插入LoRA
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# SFT配置
sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=10,
    save_steps=500,
    max_length=2048,
    # 告诉TRL哪一列是文本字段，但其实TRL可以直接识别此类的对话格式并不需要单独来说明
    # dataset_text_field="messages",
    # 而为了只计算assistant response上的loss，应该开启下面的参数
    assistant_only_loss="True"
)

# 加载训练器
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer
)
trainer.train()
# 这里的保存，保存的是训练后的Adapter/PEFT模型状态，而不是重新保存一份完整的7B Base Model
trainer.save_model("./sft_final")

In [ ]:
# SFT结束后，让我们来调用新模型
# 首先要调用基础模型
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B",
    torch_dtype="auto",
    device_map="auto"
)

# 然后加载对应Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-7B"
)

# 然后加载 SFT 过程中训练得到的 LoRA Adapter 参数
model = PeftModel.from_pretrained(
    base_model, "./sft_final"
)

model.eval()

# 构造对话
messages = [
    {
        "role": "user",
        "content": "请解释一下什么是注意力机制。"
    }
]

# 使用模型对应的Chat Template
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt=True
)

# 将对话Tokenizer编码
inputs = tokenizer(
    text, 
    return_tensors="pt"
).to(model.device)

# 模型生成
with torch.inference_model():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# 只保留模型新生成的部分
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

# 解码为文本
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)